# Tập 0.3 · ROS 2: hệ thần kinh của robot 🧠

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap03/tap03_ros_mini.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 0 · ⏱ ~30 phút

ROS 2 cho các chương trình của robot (node) nói chuyện qua kênh có tên (topic). Notebook này không cần cài ROS:
bạn tự viết một "ROS mini" bằng Python thuần để thấy cơ chế bên trong, rồi so với output của ROS 2 thật trong video.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap03")
sys.path[:0] = ["bai_tap", "sim", ".."]

from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
# Output ROS 2 Jazzy chạy thật trong video (demo.sh)
print(open("demo-output.txt", encoding="utf8").read())

## 2. Bài tập

| Bài | Viết trong | Việc |
|---|---|---|
| **1** | `BangTin.dang_ky`, `BangTin.phat` | lưu callback theo tên topic; phát tin thì gọi mọi callback của topic đó |
| **2** | `RobotNode` | đăng ký nghe `/cmd_vel`; mỗi `tick()` chạy `step()` rồi phát `/pose` |
| **3** | `DungNuaVong` | nghe `/pose`; khi θ ≥ π thì phát lệnh dừng `(0, 0)` lên `/cmd_vel` (một lần) |

Nhớ thay cả các dòng `# TODO` (đăng ký nghe topic) trong `__init__`, không chỉ các dòng `raise`.

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import math

dt = 0.01  # 100 lần mỗi giây, như robot_node.py


def step(x, y, theta, v, omega):
    """Giữ nguyên từ tập 0.2."""
    return x + v * math.cos(theta) * dt, y + v * math.sin(theta) * dt, theta + omega * dt


class BangTin:
    """Bài 1: "bảng tin" nối các node với nhau qua topic."""

    def __init__(self):
        self.nguoi_nghe = {}  # tên topic -> list các hàm callback

    def dang_ky(self, topic, callback):
        """Ghi nhận: mỗi khi có tin trên `topic` thì gọi callback(tin)."""
        raise NotImplementedError("TODO Bài 1: thêm callback vào danh sách của topic")

    def phat(self, topic, tin):
        """Gửi `tin` tới TẤT CẢ callback đã đăng ký topic này (topic chưa ai nghe thì thôi, không lỗi)."""
        raise NotImplementedError("TODO Bài 1: gọi từng callback")


class RobotNode:
    """Bài 2: node robot giống robot_node.py: nghe "/cmd_vel" (tin là tuple (v, ω)), phát "/pose" (tuple (x, y, θ))."""

    def __init__(self, bang_tin):
        self.bang_tin = bang_tin
        self.x = self.y = self.theta = 0.0
        self.v = self.omega = 0.0
        # TODO Bài 2: đăng ký nghe "/cmd_vel" bằng self.on_cmd

    def on_cmd(self, tin):
        """Nhận lệnh (v, ω), lưu lại để dùng ở các bước sau."""
        raise NotImplementedError("TODO Bài 2: lưu self.v, self.omega")

    def tick(self):
        """Gọi 100 lần mỗi giây: tiến 1 bước bằng step(), rồi phát vị trí mới lên "/pose"."""
        raise NotImplementedError("TODO Bài 2: step rồi phát /pose")


class DungNuaVong:
    """Bài 3: node điều khiển: nghe "/pose"; khi robot đã quay được nửa vòng (θ ≥ π) thì phát lệnh dừng (0, 0) lên "/cmd_vel".

    Chỉ phát lệnh dừng MỘT lần (dùng self.da_dung để nhớ)."""

    def __init__(self, bang_tin):
        self.bang_tin = bang_tin
        self.da_dung = False
        # TODO Bài 3: đăng ký nghe "/pose" bằng self.on_pose

    def on_pose(self, tin):
        raise NotImplementedError("TODO Bài 3: θ ≥ π và chưa dừng thì phát (0.0, 0.0) lên /cmd_vel")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
import loi_giai
try:
    ve(globals(), "ket_qua.png")
except NotImplementedError:
    print("Chưa xong cả 3 bài, vẽ bằng lời giải.")
    ve(loi_giai, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Thêm một node thứ ba cùng nghe `/pose` (ví dụ đếm số tin). Node robot có phải sửa gì không?

In [ ]:
import loi_giai
bt = loi_giai.BangTin()
robot = loi_giai.RobotNode(bt)
dem = {"so_tin": 0}
bt.dang_ky("/pose", lambda tin: dem.update(so_tin=dem["so_tin"] + 1))   # node "ghi_log" mới
bt.phat("/cmd_vel", (0.5, 1.0))
for _ in range(100):
    robot.tick()
print(f"node ghi_log đã nhận {dem['so_tin']} tin /pose; robot ở ({robot.x:.3f}, {robot.y:.3f})")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap03).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)